# M02c · Échantillons et estimation — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m02c_echantillons_estimation/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m02c_echantillons_estimation/03_notebook.ipynb`, créée par `python tools/start_chapter.py M02c`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M02c.1 à M02c.10, M02c.13, M02c.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M02c.18 | La distribution des moyennes d'échantillons · `sample_means` | 🔨 | ★ | 20 |
|  | M02c.19 | Biais et erreur quadratique d'un estimateur, par simulation · `estimator_bias, estimator_mse` | 🔨 | ★★ | 25 |
|  | M02c.20 | Erreur type et intervalle de confiance · `standard_error, confidence_interval_mean` | 🔨 | ★ | 20 |
|  | M02c.21 | NumPy et SciPy : `ddof`, `np.percentile`, `scipy.stats.sem` | 📦 | ★ | 15 |
| B | M02c.22 | Prédire : l'histogramme des moyennes d'une loi exponentielle quand n grandit | 🔮 | ★ | 10 |
|  | M02c.23 | Couverture : 95 % des intervalles contiennent-ils la vraie moyenne ? | 🔬 | ★★ | 20 |
|  | M02c.24 | Grande dimension : le cube dont le centre est vide | 🔬 | ★★ | 20 |
|  | M02c.25 | Voir le théorème central limite : histogrammes et densité normale | 🎨 | ★ | 20 |
| C | M02c.26 | Chasse à l'erreur : un intervalle de confiance mal calculé | 🐛 | ★ | 15 |
|  | M02c.27 | Tester une fonction aléatoire : une tolérance en erreurs types | 🛠️ | ★ | 15 |
|  | M02c.28 | Défi : un intervalle à 95 % avec dix valeurs d'une loi asymétrique | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02c")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : statistiques descriptives, valeurs aberrantes, imputation, tirages sans remise, taille d'échantillon, théorème central limite, estimateurs, intervalles de confiance, covariance et matrice de covariance, percentile de latence, standardisation.
- Écrire les cinq fonctions de `mymath/estimation.py` et les tester, puis utiliser NumPy et SciPy.
- Prédire la distribution des moyennes, mesurer la couverture d'un intervalle et le volume en grande dimension, voir le théorème central limite, chasser des erreurs d'intervalle, tester une fonction aléatoire, construire un intervalle exact.

**Rappel express.** $\bar{x} = \frac{1}{n}\sum x_i$ ; $s^2 = \frac{1}{n - 1}\sum (x_i - \bar{x})^2$ (`ddof=1`) ; $\mathbb{E}[\bar{X}] = \mu$, $\sigma(\bar{X}) = \frac{\sigma}{\sqrt{n}}$ ; pour $n$ grand, $\bar{X} \approx \mathcal{N}\big(\mu, \frac{\sigma^2}{n}\big)$ ; $\text{EQM} = \text{biais}^2 + \text{variance}$ ; intervalle à 95 % : $\bar{x} \pm 1{,}96\,\frac{s}{\sqrt{n}}$ ; $\mathbb{V}(X + Y) = \mathbb{V}(X) + \mathbb{V}(Y) + 2\operatorname{Cov}(X, Y)$.

## Partie 0 · Vérifier tes exercices papier (✏️ M02c.1 à M02c.10, 🔗 M02c.13 et M02c.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une liste `[0.25, 0.75]`, `True` / `False`, ou une lettre entre guillemets, `"B"`.

Pour la loi normale, utilise la table de $\Phi$ de M02b (§3) : les réponses attendues sont arrondies de sorte que la table et un calcul exact donnent la même valeur. Les quantiles d'une série suivent la convention de la fiche (§1). Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M02c.1 — Les statistiques d'une petite série : moyenne, médiane, quartiles, variance**

In [ ]:
# Ex M02c.1: the values you found on paper
answer_M02c_1a = ...  # a) a number (2 decimals)
answer_M02c_1b = ...  # b) a number (2 decimals)
answer_M02c_1c = ...  # c) an integer
answer_M02c_1d = ...  # d) a number (2 decimals)
answer_M02c_1e = ...  # e) a number (3 decimals)
answer_M02c_1f = ...  # f) a number (2 decimals)
answer_M02c_1g = ...  # g) a number (2 decimals)
answer_M02c_1h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M02c_1a, answer_M02c_1b, answer_M02c_1c, answer_M02c_1d, answer_M02c_1e, answer_M02c_1f, answer_M02c_1g, answer_M02c_1h]):
    wbm.check(f"M02c.1{letter}", answer)

**Ex M02c.2 — z-scores, écart interquartile et valeurs aberrantes**

In [ ]:
# Ex M02c.2: the values you found on paper
answer_M02c_2a = ...  # a) a number (1 decimal)
answer_M02c_2b = ...  # b) a number (3 decimals)
answer_M02c_2c = ...  # c) a number (2 decimals)
answer_M02c_2d = ...  # d) True or False
answer_M02c_2e = ...  # e) a number (2 decimals)
answer_M02c_2f = ...  # f) a number (3 decimals)
answer_M02c_2g = ...  # g) True or False
answer_M02c_2h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M02c_2a, answer_M02c_2b, answer_M02c_2c, answer_M02c_2d, answer_M02c_2e, answer_M02c_2f, answer_M02c_2g, answer_M02c_2h]):
    wbm.check(f"M02c.2{letter}", answer)

**Ex M02c.3 — Imputer par la moyenne ou la médiane : ce qui change**

In [ ]:
# Ex M02c.3: the values you found on paper
answer_M02c_3a = ...  # a) a number (2 decimals)
answer_M02c_3b = ...  # b) a number (2 decimals)
answer_M02c_3c = ...  # c) a number (2 decimals)
answer_M02c_3d = ...  # d) a number (2 decimals)
answer_M02c_3e = ...  # e) a number (2 decimals)
answer_M02c_3f = ...  # f) a number (3 decimals)
answer_M02c_3g = ...  # g) a number (2 decimals)
answer_M02c_3h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M02c_3a, answer_M02c_3b, answer_M02c_3c, answer_M02c_3d, answer_M02c_3e, answer_M02c_3f, answer_M02c_3g, answer_M02c_3h]):
    wbm.check(f"M02c.3{letter}", answer)

**Ex M02c.4 — Avec ou sans remise : la variance de la moyenne**

In [ ]:
# Ex M02c.4: the values you found on paper
answer_M02c_4a = ...  # a) an integer
answer_M02c_4b = ...  # b) an integer
answer_M02c_4c = ...  # c) an integer
answer_M02c_4d = ...  # d) an integer
answer_M02c_4e = ...  # e) an integer
answer_M02c_4f = ...  # f) an integer
answer_M02c_4g = ...  # g) an irreducible fraction, as text
answer_M02c_4h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M02c_4a, answer_M02c_4b, answer_M02c_4c, answer_M02c_4d, answer_M02c_4e, answer_M02c_4f, answer_M02c_4g, answer_M02c_4h]):
    wbm.check(f"M02c.4{letter}", answer)

**Ex M02c.5 — Combien d'observations pour une moyenne précise à 1 % ?**

In [ ]:
# Ex M02c.5: the values you found on paper
answer_M02c_5a = ...  # a) a number (2 decimals)
answer_M02c_5b = ...  # b) a number (2 decimals)
answer_M02c_5c = ...  # c) an integer
answer_M02c_5d = ...  # d) a number (3 decimals)
answer_M02c_5e = ...  # e) an integer
answer_M02c_5f = ...  # f) an integer
answer_M02c_5g = ...  # g) a number (2 decimals)
answer_M02c_5h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02c_5a, answer_M02c_5b, answer_M02c_5c, answer_M02c_5d, answer_M02c_5e, answer_M02c_5f, answer_M02c_5g, answer_M02c_5h]):
    wbm.check(f"M02c.5{letter}", answer)

**Ex M02c.6 — Le théorème central limite à la main : une somme de 100 variables**

In [ ]:
# Ex M02c.6: the values you found on paper
answer_M02c_6a = ...  # a) a number (2 decimals)
answer_M02c_6b = ...  # b) a number (2 decimals)
answer_M02c_6c = ...  # c) a number (1 decimal)
answer_M02c_6d = ...  # d) a number (1 decimal)
answer_M02c_6e = ...  # e) a number (4 decimals)
answer_M02c_6f = ...  # f) a number (3 decimals)
answer_M02c_6g = ...  # g) a number (2 decimals)
answer_M02c_6h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02c_6a, answer_M02c_6b, answer_M02c_6c, answer_M02c_6d, answer_M02c_6e, answer_M02c_6f, answer_M02c_6g, answer_M02c_6h]):
    wbm.check(f"M02c.6{letter}", answer)

**Ex M02c.7 — Biais, variance et erreur quadratique de trois estimateurs**

In [ ]:
# Ex M02c.7: the values you found on paper
answer_M02c_7a = ...  # a) a number (2 decimals)
answer_M02c_7b = ...  # b) an integer
answer_M02c_7c = ...  # c) an irreducible fraction, as text
answer_M02c_7d = ...  # d) a number (4 decimals)
answer_M02c_7e = ...  # e) an integer (1, 2 or 3)
answer_M02c_7f = ...  # f) a number (4 decimals)
answer_M02c_7g = ...  # g) an integer (1, 2 or 3)
answer_M02c_7h = ...  # h) a number (3 decimals)

for letter, answer in zip("abcdefgh", [answer_M02c_7a, answer_M02c_7b, answer_M02c_7c, answer_M02c_7d, answer_M02c_7e, answer_M02c_7f, answer_M02c_7g, answer_M02c_7h]):
    wbm.check(f"M02c.7{letter}", answer)

**Ex M02c.8 — Erreur type et intervalle de confiance d'une moyenne**

In [ ]:
# Ex M02c.8: the values you found on paper
answer_M02c_8a = ...  # a) a number (2 decimals)
answer_M02c_8b = ...  # b) a list [low, high] (2 decimals)
answer_M02c_8c = ...  # c) a number (2 decimals)
answer_M02c_8d = ...  # d) a number (2 decimals)
answer_M02c_8e = ...  # e) an integer
answer_M02c_8f = ...  # f) a number (3 decimals)
answer_M02c_8g = ...  # g) a number (4 decimals)
answer_M02c_8h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M02c_8a, answer_M02c_8b, answer_M02c_8c, answer_M02c_8d, answer_M02c_8e, answer_M02c_8f, answer_M02c_8g, answer_M02c_8h]):
    wbm.check(f"M02c.8{letter}", answer)

**Ex M02c.9 — La variance d'une somme et la covariance : V(X + Y), V(X − Y)**

In [ ]:
# Ex M02c.9: the values you found on paper
answer_M02c_9a = ...  # a) a number (2 decimals)
answer_M02c_9b = ...  # b) an integer
answer_M02c_9c = ...  # c) an integer
answer_M02c_9d = ...  # d) an integer
answer_M02c_9e = ...  # e) an integer
answer_M02c_9f = ...  # f) a number (2 decimals)
answer_M02c_9g = ...  # g) a number (3 decimals)
answer_M02c_9h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M02c_9a, answer_M02c_9b, answer_M02c_9c, answer_M02c_9d, answer_M02c_9e, answer_M02c_9f, answer_M02c_9g, answer_M02c_9h]):
    wbm.check(f"M02c.9{letter}", answer)

**Ex M02c.10 — La matrice de covariance de trois variables : la lire et la vérifier**

In [ ]:
# Ex M02c.10: the values you found on paper
answer_M02c_10a = ...  # a) an integer
answer_M02c_10b = ...  # b) an integer
answer_M02c_10c = ...  # c) an irreducible fraction, as text
answer_M02c_10d = ...  # d) True or False
answer_M02c_10e = ...  # e) an integer
answer_M02c_10f = ...  # f) an integer
answer_M02c_10g = ...  # g) a letter, as text
answer_M02c_10h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02c_10a, answer_M02c_10b, answer_M02c_10c, answer_M02c_10d, answer_M02c_10e, answer_M02c_10f, answer_M02c_10g, answer_M02c_10h]):
    wbm.check(f"M02c.10{letter}", answer)

**Ex M02c.13 — Le 99e percentile d'une latence : l'estimer et dire son incertitude**

In [ ]:
# Ex M02c.13: the values you found on paper
answer_M02c_13a = ...  # a) a number (2 decimals)
answer_M02c_13b = ...  # b) a number (2 decimals)
answer_M02c_13c = ...  # c) a number (3 decimals)
answer_M02c_13d = ...  # d) an integer
answer_M02c_13e = ...  # e) a number (3 decimals)
answer_M02c_13f = ...  # f) a number (2 decimals)
answer_M02c_13g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M02c_13a, answer_M02c_13b, answer_M02c_13c, answer_M02c_13d, answer_M02c_13e, answer_M02c_13f, answer_M02c_13g]):
    wbm.check(f"M02c.13{letter}", answer)

**Ex M02c.14 — Standardiser des features avec les statistiques du seul train**

In [ ]:
# Ex M02c.14: the values you found on paper
answer_M02c_14a = ...  # a) a number (2 decimals)
answer_M02c_14b = ...  # b) a number (3 decimals)
answer_M02c_14c = ...  # c) a number (3 decimals)
answer_M02c_14d = ...  # d) a number (3 decimals)
answer_M02c_14e = ...  # e) a number (3 decimals)
answer_M02c_14f = ...  # f) a number (3 decimals)
answer_M02c_14g = ...  # g) a number (2 decimals)

for letter, answer in zip("abcdefg", [answer_M02c_14a, answer_M02c_14b, answer_M02c_14c, answer_M02c_14d, answer_M02c_14e, answer_M02c_14f, answer_M02c_14g]):
    wbm.check(f"M02c.14{letter}", answer)

## Partie A · Ta librairie : moyennes d'échantillons, biais, erreur type et intervalle de confiance

Fiche §3 à §7. Les cinq fonctions de `mymath/estimation.py`, puis les mêmes statistiques avec NumPy et SciPy.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from wbm.nbtools import checked, drawn, run_pytest_group, verdict


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M02c.18 — La distribution des moyennes d'échantillons · `sample_means` 🔨 ★ ⏱️ 20 min
**Objectif :** simuler beaucoup d'échantillons, calculer leurs moyennes, et vérifier la loi des grands nombres et le théorème central limite sur ces moyennes.  
**Prérequis :** fiche §3 à §5 · **mymath :** `estimation.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M02b). Ouvre ta copie `mon_travail/mymath/estimation.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis (importe-le en haut du fichier), `scipy.stats` non : les tests comparent tes résultats à lui, et un test vérifie que tu ne t'en sers pas. Le hasard vient toujours d'**un seul** générateur `np.random.default_rng(seed)`, créé une fois au début de la fonction, et utilisé dans l'ordre que dit la docstring : c'est ce qui rend tes résultats identiques à ceux de la référence. Chaque nombre renvoyé est un `float` de Python (`float(...)` convertit un nombre de NumPy).

Écris `sample_means(draw, n, repeats, seed=0)` : un générateur `rng = np.random.default_rng(seed)`, puis `repeats` appels `draw(rng, n)`, **l'un après l'autre**, et la moyenne de chaque échantillon, dans un array NumPy de forme `(repeats,)`. `draw` peut renvoyer une liste ou un array. $n < 1$ ou `repeats` $< 1$ lèvent une `ValueError`.

Dans la vérification, `expo` tire $n$ valeurs de la loi exponentielle d'espérance 2 (`rng.exponential(2.0, n)`, d'écart-type 2 aussi), et `die` $n$ lancers d'un dé à 6 faces. La vérification calcule :
- a) la liste des 3 moyennes de `sample_means(die, 4, 3, seed=5)` (2 décimales) ;
- b) la longueur de `sample_means(expo, 10, 500)` (un entier) ;
- c) la moyenne des 20 000 moyennes de `sample_means(expo, 5, 20_000)` (2 décimales) ;
- d) leur écart-type (`np.std` ; 2 décimales) ;
- e) l'écart-type des 20 000 moyennes de 20 valeurs, divisé par celui de d) (2 décimales) ;
- f) le nom de l'erreur levée par `sample_means(expo, 3, 0)` ;
- g) le nom du type du résultat, en texte (`type(...).__name__`) ;
- h) la proportion des 20 000 moyennes de 30 valeurs qui tombent dans $2 \pm 1{,}96\,\frac{2}{\sqrt{30}}$ (2 décimales).

Dans ta copie : compare c), d), e) et h) à ce que prévoient les §3 et §5 de la fiche ($\mu$, $\frac{\sigma}{\sqrt{n}}$, $\sqrt{\frac{5}{20}}$, 0,95).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02c")  # reload your saved file

In [ ]:
expo = lambda rng, n: rng.exponential(2.0, n)      # mean 2, standard deviation 2
die = lambda rng, n: rng.integers(1, 7, n)          # n throws of a six-sided die
with checked("M02c.18a"):
    wbm.check("M02c.18a", list(mymath.estimation.sample_means(die, 4, 3, seed=5)), computed=True)
with checked("M02c.18b"):
    wbm.check("M02c.18b", len(mymath.estimation.sample_means(expo, 10, 500)), computed=True)
with checked("M02c.18c"):
    wbm.check("M02c.18c", np.mean(mymath.estimation.sample_means(expo, 5, 20_000)), computed=True)
with checked("M02c.18d"):
    wbm.check("M02c.18d", np.std(mymath.estimation.sample_means(expo, 5, 20_000)), computed=True)
with checked("M02c.18e"):
    ratio = np.std(mymath.estimation.sample_means(expo, 20, 20_000)) / np.std(
        mymath.estimation.sample_means(expo, 5, 20_000))
    wbm.check("M02c.18e", ratio, computed=True)
with checked("M02c.18f"):
    wbm.check("M02c.18f", raised(mymath.estimation.sample_means, expo, 3, 0), computed=True)
with checked("M02c.18g"):
    wbm.check("M02c.18g", type(mymath.estimation.sample_means(expo, 3, 4)).__name__, computed=True)
with checked("M02c.18h"):
    means_30 = np.asarray(mymath.estimation.sample_means(expo, 30, 20_000))
    wbm.check("M02c.18h", np.mean(np.abs(means_30 - 2) <= 1.96 * 2 / math.sqrt(30)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02c_estimation.py", "test_sample_means_ or test_estimation_uses")

### Ex M02c.19 — Biais et erreur quadratique d'un estimateur, par simulation · `estimator_bias, estimator_mse` 🔨 ★★ ⏱️ 25 min
**Objectif :** mesurer par simulation le biais et l'erreur quadratique d'un estimateur, et retrouver ceux des deux variances de l'échantillon.  
**Prérequis :** fiche §6 · **mymath :** `estimation.py` · **Parcours :** C

Écris `estimator_bias(estimator, draw_sample, true_value, repeats, seed=0)` et `estimator_mse(...)` (fiche §6) : un générateur `np.random.default_rng(seed)`, `repeats` échantillons `draw_sample(rng)` tirés l'un après l'autre, l'estimation `estimator(sample)` de chacun, puis la moyenne des estimations moins `true_value` (le biais), ou la moyenne des carrés des erreurs (l'erreur quadratique). Avec la même graine, les deux fonctions voient les mêmes échantillons : une fonction auxiliaire commune, `_estimates(...)`, évite d'écrire deux fois la simulation. `repeats` $< 1$ lève une `ValueError`.

Dans la vérification, `normal_5` tire 5 valeurs de $\mathcal{N}(0, 2^2)$ (donc $\sigma^2 = 4$), `var_0` est la variance divisée par $n$ et `var_1` la variance divisée par $n - 1$ ; `uniform_5` tire 5 valeurs uniformes sur $[0 ; 10]$. Avec `repeats = 20_000` et la graine par défaut, la vérification calcule :
- a) le biais de `var_0` (2 décimales) ;
- b) le biais de `var_1` (2 décimales) ;
- c) l'erreur quadratique de `var_0` (1 décimale) ;
- d) l'erreur quadratique de `var_1` (1 décimale) ;
- e) l'erreur quadratique de `var_0` moins le carré de son biais (1 décimale) ;
- f) le nom de l'erreur levée par `estimator_bias(var_0, normal_5, 4.0, 0)` ;
- g) le nom du type de `estimator_mse(var_0, normal_5, 4.0, 10)`, en texte ;
- h) le biais du maximum de `uniform_5` comme estimateur de la borne 10 (2 décimales).

Dans ta copie : compare a), b), c) et d) aux valeurs de la fiche ($-\frac{\sigma^2}{n}$, 0, 5,76 et 8). Que mesure e) ? Lequel des deux estimateurs préfères-tu, et selon quel critère ? h) a été calculé à la main dans la fiche : retrouves-tu la même valeur ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02c")  # reload your saved file

In [ ]:
normal_5 = lambda rng: rng.normal(0, 2, 5)        # 5 values of N(0, 2²): sigma² = 4
var_0 = lambda sample: np.var(sample)               # divided by n
var_1 = lambda sample: np.var(sample, ddof=1)       # divided by n − 1
uniform_5 = lambda rng: rng.uniform(0, 10, 5)
with checked("M02c.19a"):
    wbm.check("M02c.19a", mymath.estimation.estimator_bias(var_0, normal_5, 4.0, 20_000), computed=True)
with checked("M02c.19b"):
    wbm.check("M02c.19b", mymath.estimation.estimator_bias(var_1, normal_5, 4.0, 20_000), computed=True)
with checked("M02c.19c"):
    wbm.check("M02c.19c", mymath.estimation.estimator_mse(var_0, normal_5, 4.0, 20_000), computed=True)
with checked("M02c.19d"):
    wbm.check("M02c.19d", mymath.estimation.estimator_mse(var_1, normal_5, 4.0, 20_000), computed=True)
with checked("M02c.19e"):
    spread = (mymath.estimation.estimator_mse(var_0, normal_5, 4.0, 20_000)
              - mymath.estimation.estimator_bias(var_0, normal_5, 4.0, 20_000) ** 2)
    wbm.check("M02c.19e", spread, computed=True)
with checked("M02c.19f"):
    wbm.check("M02c.19f", raised(mymath.estimation.estimator_bias, var_0, normal_5, 4.0, 0), computed=True)
with checked("M02c.19g"):
    wbm.check("M02c.19g", type(mymath.estimation.estimator_mse(var_0, normal_5, 4.0, 10)).__name__, computed=True)
with checked("M02c.19h"):
    wbm.check("M02c.19h", mymath.estimation.estimator_bias(max, uniform_5, 10.0, 20_000), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02c_estimation.py", "test_estimator_bias_ or test_estimator_mse_")

### Ex M02c.20 — Erreur type et intervalle de confiance · `standard_error, confidence_interval_mean` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer l'erreur type d'une moyenne et son intervalle de confiance, avec leurs cas d'erreur.  
**Prérequis :** fiche §7 · **mymath :** `estimation.py` · **Parcours :** C, E

Écris `standard_error(sample, ddof=1)` puis `confidence_interval_mean(sample, z=1.96)` (fiche §7) : l'écart-type $s$ divise la somme des carrés des écarts par $n - \text{ddof}$, l'erreur type vaut $\frac{s}{\sqrt{n}}$, et l'intervalle est le **tuple** `(moyenne - z * erreur_type, moyenne + z * erreur_type)`. `standard_error` lève une `ValueError` si l'échantillon a `ddof` valeurs ou moins ; `confidence_interval_mean`, s'il a moins de deux valeurs ou si $z \le 0$.

La vérification utilise l'échantillon `[12, 15, 11, 18, 14]` et calcule :
- a) son erreur type (4 décimales) ;
- b) son erreur type avec `ddof=0` (4 décimales) ;
- c) son intervalle à 95 %, en liste (4 décimales) ;
- d) son intervalle à 90 % ($z = 1{,}645$), en liste (4 décimales) ;
- e) le nom de l'erreur levée par `standard_error([5.0])` ;
- f) le nom de l'erreur levée par `confidence_interval_mean([1, 2, 3], -1)` ;
- g) le nom du type de l'intervalle, en texte ;
- h) la largeur de l'intervalle à 95 % de 400 valeurs tirées de $\mathcal{N}(50, 10^2)$ (fournies ; 2 décimales).

Dans ta copie : refais a) et c) à la main. h) est proche de $2 \times 1{,}96 \times \frac{10}{\sqrt{400}}$ : pourquoi pas exactement ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02c")  # reload your saved file

In [ ]:
sample_20 = [12, 15, 11, 18, 14]
data_20 = np.random.default_rng(20).normal(50, 10, 400)   # 400 values of N(50, 10²)
with checked("M02c.20a"):
    wbm.check("M02c.20a", mymath.estimation.standard_error(sample_20), computed=True)
with checked("M02c.20b"):
    wbm.check("M02c.20b", mymath.estimation.standard_error(sample_20, ddof=0), computed=True)
with checked("M02c.20c"):
    wbm.check("M02c.20c", list(mymath.estimation.confidence_interval_mean(sample_20)), computed=True)
with checked("M02c.20d"):
    wbm.check("M02c.20d", list(mymath.estimation.confidence_interval_mean(sample_20, 1.645)), computed=True)
with checked("M02c.20e"):
    wbm.check("M02c.20e", raised(mymath.estimation.standard_error, [5.0]), computed=True)
with checked("M02c.20f"):
    wbm.check("M02c.20f", raised(mymath.estimation.confidence_interval_mean, [1, 2, 3], -1), computed=True)
with checked("M02c.20g"):
    wbm.check("M02c.20g", type(mymath.estimation.confidence_interval_mean(sample_20)).__name__, computed=True)
with checked("M02c.20h"):
    low, high = mymath.estimation.confidence_interval_mean(data_20)
    wbm.check("M02c.20h", high - low, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02c_estimation.py", "test_standard_error_ or test_confidence_interval_mean_")

### Ex M02c.21 — NumPy et SciPy : `ddof`, `np.percentile`, `scipy.stats.sem` 📦 ★ ⏱️ 15 min
**Objectif :** calculer avec NumPy et SciPy les statistiques d'une série, en choisissant le bon `ddof`, la bonne échelle et la bonne méthode de quantile.  
**Prérequis :** fiche §1, §2 et §7 · **Parcours :** C

Les fonctions de NumPy et de SciPy calculent les statistiques du chapitre en une ligne, à condition de lire leurs paramètres : `np.std(x, ddof=...)`, `np.percentile(x, q, method=...)` (q **en pourcentage**, de 0 à 100), `np.quantile(x, q)` (q **entre 0 et 1**), `stats.sem(x, ddof=...)` (l'erreur type de la moyenne), `stats.iqr(x)` (l'écart interquartile). Lis `help(np.percentile)` et `help(stats.sem)` avant de commencer.

La cellule fournie définit `DATA_21`, les durées d'exécution (en ms) d'un test unitaire lancé 15 fois. Calcule :
- a) l'écart-type qui divise par $n - 1$ (3 décimales) ;
- b) l'erreur type de la moyenne, avec `stats.sem` (3 décimales) ;
- c) le percentile 90, avec la méthode par défaut (2 décimales) ;
- d) le percentile 90, avec `method="nearest"` (2 décimales) ;
- e) l'écart interquartile, avec `stats.iqr` (2 décimales) ;
- f) le nombre de valeurs signalées par la règle de Tukey (un entier) ;
- g) le quantile d'ordre 0,9, avec `np.quantile` (2 décimales) ;
- h) `stats.sem(DATA_21, ddof=0) / stats.sem(DATA_21)` (4 décimales).

Dans ta copie : retrouve c) avec la convention de la fiche (§1), et h) par une formule en $n$. Pourquoi c) et d) diffèrent-ils ?

In [ ]:
DATA_21 = np.array([23, 19, 31, 25, 22, 27, 24, 88, 21, 26, 30, 20, 24, 29, 23])   # 15 durations, in ms

In [ ]:
answer_M02c_21a = ...  # a) np.std(...)
answer_M02c_21b = ...  # b) stats.sem(...)
answer_M02c_21c = ...  # c) np.percentile(...)
answer_M02c_21d = ...  # d) np.percentile(..., method="nearest")
answer_M02c_21e = ...  # e) stats.iqr(...)
answer_M02c_21f = ...  # f) an integer
answer_M02c_21g = ...  # g) np.quantile(...)
answer_M02c_21h = ...  # h) stats.sem(DATA_21, ddof=0) / stats.sem(DATA_21)

In [ ]:
for letter, answer in zip("abcdefgh", [answer_M02c_21a, answer_M02c_21b, answer_M02c_21c, answer_M02c_21d,
                                        answer_M02c_21e, answer_M02c_21f, answer_M02c_21g, answer_M02c_21h]):
    wbm.check(f"M02c.21{letter}", answer)

## Partie B · Une prédiction, deux expériences et une figure

Fiche §4, §5, §7 et §9. Les moyennes d'une loi asymétrique, la couverture d'un intervalle, le volume en grande dimension, le théorème central limite en images.

### Ex M02c.22 — Prédire : l'histogramme des moyennes d'une loi exponentielle quand n grandit 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir comment le centre, la largeur et la forme de la distribution des moyennes changent avec $n$, puis le vérifier par simulation.  
**Prérequis :** fiche §4 et §5 · **Parcours :** C, E

On tire beaucoup d'échantillons de $n$ valeurs d'une loi exponentielle d'espérance 1 (et d'écart-type 1), et on trace l'histogramme de leurs moyennes, pour $n = 1, 4, 16, 64$. Choisis, pour chaque question :
- a) quand $n$ passe de 4 à 16, l'écart-type des moyennes est : (1) le même ; (2) divisé par 2 ; (3) divisé par 4 ; (4) divisé par 16 ;
- b) quand $n$ grandit, le centre de l'histogramme (la moyenne des moyennes) : (1) reste en 1 ; (2) se rapproche de 0 ; (3) se rapproche de la médiane de la loi, $\ln 2 \approx 0{,}69$ ; (4) grandit ;
- c) pour $n = 1$, l'histogramme : (1) a la forme d'une cloche symétrique ; (2) part d'un maximum en 0 et décroît, avec une longue queue à droite ; (3) est plat ; (4) a une longue queue à gauche ;
- d) pour $n = 1$, la médiane des tirages, comparée à leur moyenne, est : (1) égale ; (2) plus petite ; (3) plus grande ; (4) impossible à prévoir.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**, de 1 à 4), vérifie-les, puis lance l'expérience : 100 000 échantillons pour chaque $n$.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M02c_22a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M02c_22b = ...  # b) 1, 2, 3 or 4
prediction_M02c_22c = ...  # c) 1, 2, 3 or 4
prediction_M02c_22d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M02c_22a, prediction_M02c_22b, prediction_M02c_22c,
                                      prediction_M02c_22d]):
    wbm.check(f"M02c.22{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M02c_22a, prediction_M02c_22b, prediction_M02c_22c, prediction_M02c_22d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    rng_22 = np.random.default_rng(22)
    fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
    for ax, n in zip(axes, [1, 4, 16, 64]):
        means = rng_22.exponential(1.0, (100_000, n)).mean(axis=1)    # 100 000 means of n values, mean 1, std 1
        ax.hist(means, bins=80, density=True, alpha=0.6)
        ax.axvline(1, color="black", linewidth=1)
        ax.set_title(f"n = {n}")
        print(f"n = {n:2d} : moyenne {means.mean():.3f} · écart-type {means.std():.3f} · médiane {np.median(means):.3f}")
    plt.show()

**Après l'expérience**, dans ta copie : explique chaque résultat avec la fiche ($\mathbb{E}[\bar{X}_n]$, $\sigma(\bar{X}_n)$, le théorème central limite). Pour $n = 64$, la cloche est-elle parfaitement symétrique ? Que deviennent la médiane et la moyenne des moyennes quand $n$ grandit ?

### Ex M02c.23 — Couverture : 95 % des intervalles contiennent-ils la vraie moyenne ? 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer par simulation la couverture de l'intervalle $\bar{x} \pm z\,\frac{s}{\sqrt{n}}$, et voir quand elle s'éloigne de 95 %.  
**Prérequis :** fiche §5 et §7 · **Parcours :** C, E

La **couverture** d'une méthode d'intervalle de confiance est la proportion des échantillons pour lesquels l'intervalle contient la vraie valeur (fiche §7). La cellule fournie tire, une fois pour toutes, trois paquets de 10 000 échantillons (une ligne par échantillon) : `NORMAL_20` (20 valeurs de $\mathcal{N}(10, 3^2)$), `EXPO_10` et `EXPO_100` (10 et 100 valeurs d'une loi exponentielle d'espérance 2).

Écris `coverage(samples, true_mean, z=1.96)`, qui calcule pour **chaque ligne** l'intervalle $\bar{x} \pm z\,\frac{s}{\sqrt{n}}$ (avec $s$ divisé par $n - 1$) et renvoie la proportion des lignes dont l'intervalle contient `true_mean`. Vectorise : `samples.mean(axis=1)` et `samples.std(axis=1, ddof=1)` traitent toutes les lignes d'un coup.

La vérification appelle ta fonction et calcule (3 décimales) :
- a) la couverture à 95 % pour `NORMAL_20` ;
- b) pour `EXPO_10` ;
- c) pour `EXPO_100` ;
- d) la couverture pour `NORMAL_20` avec $z = 1{,}645$ ;
- e) la couverture pour `NORMAL_20` avec le quantile de Student à 19 degrés de liberté, `stats.t.ppf(0.975, 19)`, à la place de 1,96.

Dans ta copie : l'aléa d'une proportion estimée sur 10 000 échantillons est d'environ $\sqrt{\frac{0{,}95 \times 0{,}05}{10\,000}} \approx 0{,}002$. Lesquelles de ces couvertures sont nettement sous 0,95 ? Pourquoi b) est-elle la plus basse, et pourquoi c) remonte-t-elle ? Que corrige e) ?

In [ ]:
rng_23 = np.random.default_rng(23)
NORMAL_20 = rng_23.normal(10, 3, (10_000, 20))         # 10 000 samples of 20 values of N(10, 3²): true mean 10
EXPO_10 = rng_23.exponential(2.0, (10_000, 10))        # 10 000 samples of 10 values, exponential law: true mean 2
EXPO_100 = rng_23.exponential(2.0, (10_000, 100))      # 10 000 samples of 100 values of the same law

In [ ]:
def coverage(samples, true_mean, z=1.96):
    """The share of the rows of samples whose interval mean ± z s/√n (s with ddof=1) contains true_mean."""
    raise NotImplementedError("une moyenne et un écart-type par ligne, puis la proportion des lignes qui conviennent")

In [ ]:
with checked("M02c.23a"):
    wbm.check("M02c.23a", coverage(NORMAL_20, 10), computed=True)
with checked("M02c.23b"):
    wbm.check("M02c.23b", coverage(EXPO_10, 2), computed=True)
with checked("M02c.23c"):
    wbm.check("M02c.23c", coverage(EXPO_100, 2), computed=True)
with checked("M02c.23d"):
    wbm.check("M02c.23d", coverage(NORMAL_20, 10, z=1.645), computed=True)
with checked("M02c.23e"):
    wbm.check("M02c.23e", coverage(NORMAL_20, 10, z=stats.t.ppf(0.975, 19)), computed=True)

### Ex M02c.24 — Grande dimension : le cube dont le centre est vide 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer par Monte-Carlo la part du cube occupée par la boule inscrite et par une fine couche près du bord, et voir où la simulation cesse d'être utile.  
**Prérequis :** fiche §4 et §9 · **Parcours :** C

La cellule fournie tire 100 000 points uniformes dans le cube $[-1 ; 1]^d$ pour $d = 2, 3, 10, 20$ (le dictionnaire `CUBES_24`, une ligne par point). Écris :
- `ball_fraction(points)` : la proportion des points dans la boule de rayon 1 centrée en 0 ($x_1^2 + \cdots + x_d^2 \le 1$) ;
- `border_fraction(points, eps=0.05)` : la proportion des points à moins de `eps` du bord du cube, c'est-à-dire dont **au moins une** coordonnée dépasse $1 - \varepsilon$ en valeur absolue ;
- `exact_ball_fraction(d)` : la valeur exacte, $\frac{\pi^{d/2}}{\Gamma(d/2 + 1)\,2^d}$ (la formule admise par la fiche ; `math.gamma` calcule $\Gamma$).

La vérification appelle tes fonctions et calcule :
- a) `ball_fraction` pour $d = 2$ (4 décimales) ;
- b) pour $d = 3$ (4 décimales) ;
- c) pour $d = 10$ (4 décimales) ;
- d) `border_fraction` pour $d = 20$ (3 décimales) ;
- e) `exact_ball_fraction(10)` (5 décimales) ;
- f) le **nombre** de points de `CUBES_24[20]` dans la boule : la proportion de `ball_fraction` multipliée par 100 000 (un entier).

Dans ta copie : compare a), b), d) aux valeurs exactes de la fiche ($\frac{\pi}{4}$, $\frac{\pi}{6}$, $1 - 0{,}95^{20}$). L'écart entre c) et e) est-il normal, compte tenu de l'erreur type d'une proportion ? Combien de points faudrait-il, en moyenne, pour en voir **un seul** dans la boule en dimension 20 (`exact_ball_fraction(20)`) ? Que dit f) des limites de Monte-Carlo pour un événement très rare ?

In [ ]:
rng_24 = np.random.default_rng(24)
CUBES_24 = {d: rng_24.uniform(-1, 1, (100_000, d)) for d in (2, 3, 10, 20)}   # 100 000 points in [−1, 1]^d

In [ ]:
def ball_fraction(points):
    """The share of the points (one per row) inside the ball of radius 1 centred at 0."""
    raise NotImplementedError("une condition sur chaque ligne de points, puis la proportion des lignes qui la vérifient")


def border_fraction(points, eps=0.05):
    """The share of the points with at least one coordinate beyond 1 − eps in absolute value."""
    raise NotImplementedError("« au moins une coordonnée » : quelle fonction de NumPy le dit pour chaque ligne ?")


def exact_ball_fraction(d):
    """The volume of the unit ball of dimension d divided by the volume 2^d of the cube [−1, 1]^d."""
    raise NotImplementedError("la formule de l'énoncé, avec math.pi et math.gamma")

In [ ]:
with checked("M02c.24a"):
    wbm.check("M02c.24a", ball_fraction(CUBES_24[2]), computed=True)
with checked("M02c.24b"):
    wbm.check("M02c.24b", ball_fraction(CUBES_24[3]), computed=True)
with checked("M02c.24c"):
    wbm.check("M02c.24c", ball_fraction(CUBES_24[10]), computed=True)
with checked("M02c.24d"):
    wbm.check("M02c.24d", border_fraction(CUBES_24[20]), computed=True)
with checked("M02c.24e"):
    wbm.check("M02c.24e", exact_ball_fraction(10), computed=True)
with checked("M02c.24f"):
    wbm.check("M02c.24f", round(ball_fraction(CUBES_24[20]) * len(CUBES_24[20])), computed=True)

### Ex M02c.25 — Voir le théorème central limite : histogrammes et densité normale 🎨 ★ ⏱️ 20 min
**Objectif :** tracer les histogrammes des moyennes d'échantillons pour plusieurs tailles, avec la densité normale que prévoit le théorème central limite.  
**Prérequis :** fiche §5 ; R0 (matplotlib) · **Parcours :** C

Écris `draw_clt(axes, draw, mu, sigma, ns, repeats=20_000, seed=0)`, qui dessine, pour chaque taille $n$ de la liste `ns` et dans l'axe correspondant de `axes` :
- l'**histogramme** des `repeats` moyennes de $n$ tirages (ta fonction `mymath.estimation.sample_means(draw, n, repeats, seed)`, ou ton propre code), avec `density=True` pour qu'il ait une aire de 1, comme une densité ;
- la **densité normale** prévue par le théorème central limite, $\mathcal{N}\big(\mu, \frac{\sigma^2}{n}\big)$, tracée sur une grille de valeurs (`stats.norm(mu, sigma / math.sqrt(n)).pdf(grid)`, avec `grid` couvrant au moins $\mu \pm 4\frac{\sigma}{\sqrt{n}}$) ;
- un titre qui indique $n$.

Puis trace-la pour la loi exponentielle d'espérance 1 (`EXPO_DRAW`, $\mu = \sigma = 1$) et `NS_25` $= [1, 2, 10, 50]$.

Dans ta copie : pour quel $n$ la densité normale te semble-t-elle décrire correctement l'histogramme ? Où l'écart se voit-il le plus pour $n = 10$ : au centre ou dans les queues ? Pourquoi la courbe de $n = 1$ est-elle si loin de l'histogramme ?

In [ ]:
EXPO_DRAW = lambda rng, n: rng.exponential(1.0, n)     # the exponential law of mean 1 and standard deviation 1
NS_25 = [1, 2, 10, 50]

In [ ]:
def draw_clt(axes, draw, mu, sigma, ns, repeats=20_000, seed=0):
    """For each n of ns, in the matching axis: the histogram (density=True) of repeats means of n draws, the normal
    density N(mu, sigma²/n), and a title with n."""
    raise NotImplementedError("sample_means, ax.hist(..., density=True), ax.plot(grid, stats.norm(...).pdf(grid))")

In [ ]:
with checked("M02c.25 figure"):
    fig, axes = plt.subplots(1, 4, figsize=(13, 3))
    try:
        created = drawn(draw_clt, axes, EXPO_DRAW, 1.0, 1.0, NS_25)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M02c.25 : draw_clt ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans les axes "
              "reçus.")
    else:
        fig.suptitle("Moyennes de n tirages d'une loi exponentielle, et la densité normale prévue", y=1.05)
        plt.show()

**Vérifications automatiques** (sur une autre loi, uniforme ; la figure elle-même se juge à l'œil).

In [ ]:
with checked("M02c.25 vérifications"):
    from matplotlib.patches import Polygon, Rectangle

    def histogram_bars(ax):
        """(left, width, height) of the bars of the histograms of ax: filled bars, or the steps of histtype="step"."""
        bars = [(p.get_x(), p.get_width(), p.get_height()) for p in ax.patches
                if isinstance(p, Rectangle) and p.get_height() > 0]
        for patch in ax.patches:
            if isinstance(patch, Polygon):
                xy = np.asarray(patch.get_xy(), dtype=float)
                bars += [(x0, x1 - x0, y0) for (x0, y0), (x1, y1) in zip(xy[:-1], xy[1:])
                         if y0 == y1 and y0 > 0 and x1 > x0]
        return bars

    problems = []
    uniform_draw = lambda rng, n: rng.random(n)            # another law: uniform on [0, 1], mean 0.5, std √(1/12)
    mu_t, sigma_t = 0.5, math.sqrt(1 / 12)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    try:
        created = drawn(draw_clt, axes, uniform_draw, mu_t, sigma_t, [1, 9], 4_000)
    finally:
        plt.close(fig)                                     # only the checks below are shown here
    if created:
        problems.append("draw_clt ouvre sa propre figure : dessine dans les axes reçus")
    else:
        all_bars = [histogram_bars(ax) for ax in axes]
        if not all(all_bars):
            problems.append("un axe n'a aucun histogramme : dessine chaque histogramme avec ax.hist dans l'axe reçu (pas "
                            "plt.hist, qui dessine tout dans le dernier axe)")
        for ax, n, bars in zip(axes, [1, 9], all_bars if all(all_bars) else []):
            width_t = sigma_t / math.sqrt(n)
            area = sum(w * h for _, w, h in bars)
            if abs(area - 1) > 0.02:
                problems.append(f"n = {n} : l'histogramme doit avoir une aire de 1 (density=True)")
                continue
            centre = sum((x + w / 2) * w * h for x, w, h in bars) / area
            if abs(centre - mu_t) > 0.05:
                problems.append(f"n = {n} : l'histogramme doit être celui des moyennes de n tirages, centré sur mu")
            spread = math.sqrt(sum((x + w / 2 - centre) ** 2 * w * h for x, w, h in bars) / area)
            if abs(spread / width_t - 1) > 0.15:
                problems.append(f"n = {n} : la largeur de l'histogramme doit être celle des moyennes de n tirages, "
                                "sigma / √n")
            curves = [line for line in ax.get_lines() if len(line.get_xdata()) > 20]
            if not curves:
                problems.append(f"n = {n} : il manque la courbe de la densité normale (ax.plot sur une grille)")
                continue
            expected = 1 / (width_t * math.sqrt(2 * math.pi))
            issues = []
            for line in curves:                              # one right curve is enough, whatever else is drawn
                xs = np.asarray(line.get_xdata(), dtype=float)
                ys = np.asarray(line.get_ydata(), dtype=float)
                found = []
                if abs(xs[np.argmax(ys)] - mu_t) > 0.02 * (1 + sigma_t):
                    found.append(f"n = {n} : le sommet de la densité normale doit être en mu (il est en "
                                 f"{xs[np.argmax(ys)]:.3f})")
                if abs(ys.max() / expected - 1) > 0.02:
                    found.append(f"n = {n} : la densité normale doit avoir l'écart-type sigma / √n (sa hauteur au "
                                 "sommet n'est pas la bonne)")
                if xs.min() > mu_t - 3.5 * width_t or xs.max() < mu_t + 3.5 * width_t:
                    found.append(f"n = {n} : la grille de la densité doit couvrir au moins mu ± 4 sigma / √n")
                if not found:
                    issues = []
                    break
                issues = issues or found
            problems += issues
            if str(n) not in ax.get_title():
                problems.append(f"n = {n} : le titre de chaque axe doit indiquer n")
    if problems:
        print("❌ Ex M02c.25 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M02c.25 : des histogrammes de moyennes, d'aire 1, bien centrés, avec la bonne densité normale.")

## Partie C · Erreurs classiques, tests et défi

Fiche §4 à §7. Quatre erreurs d'intervalle à trouver, le test d'une fonction aléatoire, puis un intervalle exact pour dix valeurs.

### Ex M02c.26 — Chasse à l'erreur : un intervalle de confiance mal calculé 🐛 ★ ⏱️ 15 min
**Objectif :** repérer les erreurs classiques d'une erreur type ou d'un intervalle de confiance dans du code, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §7 · **Parcours :** C

Une collègue a mesuré $n = 25$ fois la durée d'un traitement : moyenne `mean = 50`, écart-type `s = 10` (divisé par $n - 1$). Elle a aussi évalué un modèle : accuracy `acc = 0.8` sur `n_test = 400` exemples. Voici ses six lignes de code, chacune avec un commentaire qui dit ce qu'elle veut calculer. Quatre lignes sont fausses. **Sans exécuter de code**, trouve-les.

```python
half_A = 1.96 * s / n                                # (A) half-width of the 95 % interval of the mean
half_B = 1.96 * s ** 2 / math.sqrt(n)                # (B) half-width of the 95 % interval of the mean
half_C = 1.645 * s / math.sqrt(n)                    # (C) half-width of the 90 % interval of the mean
half_D = 1.96 * s / math.sqrt(n)                     # (D) half-width of the 99 % interval of the mean
low_E, high_E = mean - 1.96 * s, mean + 1.96 * s     # (E) the 95 % confidence interval of the mean
se_F = math.sqrt(acc * (1 - acc) / n_test)           # (F) standard error of the accuracy
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire : la demi-largeur demandée pour (A) à (D), la demi-largeur de l'intervalle correct pour (E), l'erreur type pour (F) (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle confusion a été faite ? L'intervalle de (E) a-t-il quand même un sens ?

In [ ]:
answer_M02c_26a = ...  # a) letters, as text
answer_M02c_26b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M02c.26a", answer_M02c_26a)
wbm.check("M02c.26b", answer_M02c_26b)

### Ex M02c.27 — Tester une fonction aléatoire : une tolérance en erreurs types 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire le test d'une fonction aléatoire avec une tolérance fixée en erreurs types, ni trop étroite (test instable) ni trop large (bugs invisibles).  
**Prérequis :** fiche §4, §5 et §7 ; Ex M02c.18 · **Parcours :** C

Comment tester une fonction dont le résultat est aléatoire ? Comparer à la valeur exacte avec `==` échoue toujours ; avec une tolérance prise au hasard (`abs(x - pi) < 0.1`), le test laisse passer des bugs, ou échoue sans bug (on dit qu'il est **instable**, *flaky*). La bonne tolérance se compte en **erreurs types** : par le théorème central limite, une estimation juste s'écarte de plus de $k$ erreurs types de la vraie valeur avec la probabilité $2\big(1 - \Phi(k)\big)$ environ.

La cellule fournie définit `estimate_pi(n, rng)`, qui estime $\pi$ par Monte-Carlo (4 fois la proportion de $n$ points uniformes du carré $[0 ; 1]^2$ tombés dans le quart de disque de rayon 1), et deux copies fausses. Écris :
- `pi_standard_error(n)` : l'erreur type de `estimate_pi` pour $n$ points (pourquoi est-ce 4 fois celle d'une proportion ?) ;
- `TOLERANCE_K` : le nombre $k$ d'erreurs types que tu tolères. Choisis-le : un vrai projet lance ses tests des milliers de fois, donc une estimation juste ne doit presque jamais échouer ($2(1 - \Phi(k))$ minuscule) ; et la copie fausse au rayon de 0,99 a un biais de 12 erreurs types environ, qu'il faut attraper à coup sûr ;
- `pi_test(estimate, seed, n=100_000)` : renvoie `True` si `estimate(n, np.random.default_rng(seed))` est à moins de `TOLERANCE_K` erreurs types de $\pi$, `False` sinon.

La vérification lance ton test sur la bonne fonction et sur la copie au rayon de 0,99, pour 100 graines : il doit accepter la bonne fonction **toutes** les fois, et refuser la fausse **toutes** les fois.

Dans ta copie : justifie ton choix de $k$. Que se passerait-il avec $k = 1$ ? Avec $k = 50$ ? Pourquoi la fonction décalée de $+0{,}002$ passe-t-elle presque toujours, et combien de points faudrait-il pour l'attraper ?

In [ ]:
def estimate_pi(n, rng):
    """Monte Carlo estimate of π: 4 × the share of n uniform points of [0, 1]² inside the quarter disc."""
    points = rng.random((n, 2))
    return 4 * float(np.mean(np.sum(points ** 2, axis=1) <= 1))


def estimate_pi_shrunk(n, rng):
    """A buggy copy: the quarter disc has a radius of 0.99 instead of 1 (an error of about 2 %)."""
    points = rng.random((n, 2))
    return 4 * float(np.mean(np.sum(points ** 2, axis=1) <= 0.99 ** 2))


def estimate_pi_slightly_off(n, rng):
    """A buggy copy whose error (+0.002) is smaller than one standard error for n = 100 000."""
    return estimate_pi(n, rng) + 0.002

In [ ]:
def pi_standard_error(n):
    """The standard error of estimate_pi for n points."""
    raise NotImplementedError("l'erreur type d'une proportion p = π/4, puis l'effet du facteur 4")


TOLERANCE_K = ...   # your choice: the number of standard errors your test tolerates


def pi_test(estimate, seed, n=100_000):
    """True if estimate(n, rng) is within TOLERANCE_K standard errors of π, with rng = np.random.default_rng(seed)."""
    raise NotImplementedError("l'écart à math.pi, comparé à une tolérance comptée en erreurs types")

In [ ]:
with checked("M02c.27 vérifications"):
    problems = []
    se = pi_standard_error(100_000)
    if not isinstance(se, (int, float)) or abs(se - 4 * math.sqrt(math.pi / 4 * (1 - math.pi / 4) / 100_000)) > 1e-6:
        problems.append("pi_standard_error(n) : l'erreur type de 4 fois une proportion (relis l'énoncé : pourquoi 4 ?)")
    if TOLERANCE_K is ...:
        raise NotImplementedError("choisis TOLERANCE_K, le nombre d'erreurs types que ton test tolère")
    if not isinstance(TOLERANCE_K, (int, float)):
        raise TypeError(f"TOLERANCE_K doit être un nombre, pas {type(TOLERANCE_K).__name__}")
    seen = []

    def spy(n, rng):                                       # records the first number of the generator it receives
        seen.append(float(rng.random()))
        return math.pi

    answer = pi_test(spy, 7)
    if not isinstance(answer, (bool, np.bool_)):
        problems.append(f"pi_test doit renvoyer True ou False, pas {type(answer).__name__} ({answer!r})")
    elif not seen or seen[0] != float(np.random.default_rng(7).random()):
        problems.append("pi_test doit appeler estimate(n, np.random.default_rng(seed)) avec la graine reçue : un test "
                        "doit être reproductible")
    else:
        passed = [bool(pi_test(estimate_pi, seed)) for seed in range(100)]
        caught = [not pi_test(estimate_pi_shrunk, seed) for seed in range(100)]
        print(f"k = {TOLERANCE_K} : la bonne fonction passe ton test pour {sum(passed)} graines sur 100 ; la fonction "
              f"fausse est attrapée {sum(caught)} fois sur 100")
        if sum(passed) < 100:
            problems.append("ton test échoue sur la bonne fonction pour certaines graines : la tolérance est trop "
                            "étroite (un test qui échoue sans bug est un test « instable », flaky)")
        if sum(caught) < 100:
            problems.append("ton test laisse passer la fonction au rayon de 0,99 : la tolérance est trop large")
        if not problems and not 3.5 <= TOLERANCE_K <= 10:
            problems.append("ces 100 graines passent, mais ce k serait instable ou trop large sur des milliers de "
                            "tests : relis l'énoncé pour le choisir")
    if problems:
        print("❌ Ex M02c.27 : à revoir : " + " ; ".join(problems) + ".")
    else:
        slight = sum(not pi_test(estimate_pi_slightly_off, seed) for seed in range(100))
        print(f"✅ Ex M02c.27 : une tolérance juste. (La fonction décalée de +0,002 n'est attrapée que {slight} fois sur "
              "100 : un écart plus petit qu'une erreur type est invisible pour un seul tirage.)")

### Ex M02c.28 — Défi : un intervalle à 95 % avec dix valeurs d'une loi asymétrique 🏆 ★★★ ⏱️ 30 min
**Objectif :** construire un intervalle de confiance à 95 % pour la moyenne d'une loi exponentielle à partir de dix valeurs seulement, avec une couverture juste et une largeur raisonnable.  
**Prérequis :** fiche §5 à §7 ; Ex M02c.23 · **Parcours :** C

Avec 10 valeurs d'une loi exponentielle, l'intervalle $\bar{x} \pm 1{,}96\,\frac{s}{\sqrt{n}}$ ne couvre la vraie moyenne qu'environ 87 fois sur 100 (M02c.23), et le quantile de Student ne suffit pas. Écris `interval_10(sample)`, qui reçoit un array de **10 valeurs** tirées d'une loi exponentielle de moyenne $\mu$ **inconnue**, et renvoie un intervalle `(low, high)` pour $\mu$. **Objectifs**, mesurés sur les 4 500 échantillons de la cellule fournie (1 500 pour chacune des moyennes 0,5, 3 et 40, que ta fonction ne connaît pas) :
1. la couverture est entre 0,94 et 0,96, et entre 0,93 et 0,97 pour chaque moyenne ;
2. la largeur moyenne des intervalles est au plus $1{,}6\,\mu$ ;
3. moins de 20 secondes pour les 4 500 intervalles.

Tous les outils sont permis (NumPy, SciPy, une simulation). Pistes (`04_indices.md` les détaille) :
- le bootstrap des percentiles (fiche §7) : tente-le, et regarde quel objectif il manque (vectorise-le : `sample[rng.integers(0, 10, (2000, 10))]` tire d'un coup les 2 000 rééchantillons ; une boucle Python sur les rééchantillons prendrait plus d'une minute ici) ;
- une remarque clé : si $X$ suit une loi exponentielle de moyenne $\mu$, alors $\frac{X}{\mu}$ suit la loi exponentielle de moyenne 1. Que peux-tu en dire de la loi de $\frac{\bar{X}}{\mu}$ ? Dépend-elle de $\mu$ ? Comment connaître ses quantiles, même sans formule ?

Dans ta copie : quelle méthode as-tu retenue ? Pourquoi ton intervalle n'est-il pas symétrique autour de $\bar{x}$ ? Pourquoi le bootstrap des percentiles couvre-t-il mal avec 10 valeurs ?

In [ ]:
rng_28 = np.random.default_rng(28)
SCALES_28 = np.repeat([0.5, 3.0, 40.0], 1500)                              # the true means μ, unknown to you
SAMPLES_28 = rng_28.exponential(1.0, (4500, 10)) * SCALES_28[:, None]      # 4 500 samples of 10 values

In [ ]:
def interval_10(sample):
    """A 95 % confidence interval (low, high) for the mean of an exponential law, from 10 values."""
    raise NotImplementedError("le bootstrap des percentiles d'abord, puis la remarque clé de l'énoncé")

In [ ]:
with checked("M02c.28"):
    import time

    start = time.perf_counter()
    intervals = np.array([interval_10(sample.copy()) for sample in SAMPLES_28], dtype=float)
    seconds = time.perf_counter() - start
    reasons = []
    if intervals.shape != (4500, 2):
        reasons.append("interval_10 doit renvoyer deux nombres (low, high)")
    else:
        inside = (intervals[:, 0] <= SCALES_28) & (SCALES_28 <= intervals[:, 1])
        overall = float(np.mean(inside))
        per_scale = [float(np.mean(inside[SCALES_28 == mu])) for mu in (0.5, 3.0, 40.0)]
        width = float(np.mean((intervals[:, 1] - intervals[:, 0]) / SCALES_28))
        print(f"couverture : {overall:.4f} (par moyenne μ = 0,5 / 3 / 40 : "
              + " / ".join(f"{c:.3f}" for c in per_scale) + f") · largeur moyenne : {width:.3f} μ · temps : {seconds:.1f} s")
        if not 0.94 <= overall <= 0.96:
            reasons.append("la couverture n'est pas entre 0,94 et 0,96")
        elif not all(0.93 <= c <= 0.97 for c in per_scale):
            reasons.append("la couverture doit tenir pour chaque valeur de μ (entre 0,93 et 0,97) : ton intervalle ne "
                           "doit pas dépendre d'une échelle choisie d'avance")
        if not width <= 1.6:
            reasons.append("les intervalles sont trop larges (plus de 1,6 μ en moyenne)")
    if seconds > 20:
        reasons.append("plus de 20 secondes pour 4 500 intervalles : vectorise avec NumPy, et calcule une seule fois "
                       "ce qui ne dépend pas de l'échantillon")
    for reason in reasons:
        print(f"❌ {reason}")
    if reasons:
        print("❌ Ex M02c.28 : un objectif au moins n'est pas atteint : relis les pistes de l'énoncé et de 04_indices.md.")
    else:
        wbm.check("M02c.28", True, computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu résumer une série, repérer ses valeurs aberrantes, et dire ce qu'une imputation change ?
2. Sais-tu utiliser la loi des grands nombres et le théorème central limite, et calculer le biais et l'erreur quadratique d'un estimateur ?
3. Sais-tu calculer et interpréter une erreur type, un intervalle de confiance, une covariance et une corrélation ?

**La suite** : le chapitre 2 de workbookIA (hasard et statistiques de base), puis M03, les probabilités conditionnelles.